In [1]:
import sys
from pathlib import Path

repo_root = Path().resolve().parents[3]
sys.path.insert(0, str(repo_root))
print(sys.path)

['C:\\Users\\sacha\\RiceLocal\\Capstone\\Coding\\Flood-Forecasting', 'C:\\Users\\sacha\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'C:\\Users\\sacha\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'C:\\Users\\sacha\\AppData\\Local\\Programs\\Python\\Python312\\Lib', 'C:\\Users\\sacha\\AppData\\Local\\Programs\\Python\\Python312', 'c:\\Users\\sacha\\RiceLocal\\Capstone\\Coding\\Flood-Forecasting\\.venv', '', 'c:\\Users\\sacha\\RiceLocal\\Capstone\\Coding\\Flood-Forecasting\\.venv\\Lib\\site-packages', 'C:\\Users\\sacha\\RiceLocal\\Capstone\\Coding\\Flood-Forecasting', 'c:\\Users\\sacha\\RiceLocal\\Capstone\\Coding\\Flood-Forecasting\\.venv\\Lib\\site-packages\\win32', 'c:\\Users\\sacha\\RiceLocal\\Capstone\\Coding\\Flood-Forecasting\\.venv\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\sacha\\RiceLocal\\Capstone\\Coding\\Flood-Forecasting\\.venv\\Lib\\site-packages\\Pythonwin']


In [ ]:
"""
LSTM Flood Forecasting Model
Predicts streamflow 24 hours ahead using the top30 high flood-severity sites.
"""
import torch
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

from src.preprocessing.preprocessing import processor
from src.models.lstm import LSTMModel
from src.utils.helpers import create_sequences

STATIC_FEATURES = [
    "longitude", "latitude", "DRAIN_SQKM", "artificial_path_pct",
    "wb5100_ann_mm", "snw_pc_syr", "snow_ice_nlcd06", "barren_nlcd06",
    "mains100_plant", "hga", "hgc", "bulk_density_avg", "elev_max_m", "aspect_deg"
]

DYNAMIC_FEATURES = [
    "streamflow_cfs_mean", "streamflow_cfs_max", "streamflow_cfs_min",
    "gage_height_ft_mean",
    "precipitation_mm",
    "temperature_c",
    "potential_evaporation_mm",
    "specific_humidity_kgkg",
    "shortwave_radiation_wm2",
    "longwave_radiation_wm2",
    "wind_speed_ms",
    "surface_pressure_pa",
    "cape_jkg",
    "convective_precip_fraction",
]

TARGET = "streamflow_cfs_mean" 

WINDOW_SIZE = 72

config = {
    "input_cols": DYNAMIC_FEATURES + STATIC_FEATURES,
    "static_cols": STATIC_FEATURES,
    "target": "streamflow_cfs_target_24h",
    "train_split": 0.8,
    "val_split": 0.9,
    "file_path": "flood-dataset-top30",
    "file_name": "flood_model_top30",
    "table": "wandb.flood_model_top30",
    "lag_window": 1,
    "frequency": "hourly",
    "split_time_days": 30,
    "site_scaling": False,
}

In [2]:
pcr = processor(config)
pcr.pull_wandb()
print(pcr.df["site_id"].unique())
print(pcr.df.shape)
train_X, val_X, test_X, train_y, val_y, test_y = pcr.return_outputs()

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\sacha\_netrc.
wandb: Downloading large artifact 'flood-dataset-top30:latest', 145.19MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:02.7 (54.3MB/s)


DuplicateError: projections contained duplicate output name 'site_id'. It's possible that multiple expressions are returning the same default column name. If this is the case, try renaming the columns with `.alias("new_name")` to avoid duplicate column names.